# Two Rings PCA Cannot Separate

**Topic:** Kernel PCA &nbsp;·&nbsp; **Difficulty:** Medium

Concentric circles are the standard counterexample to linear methods. The classes
are perfectly separable (by radius), but no linear projection of the raw
coordinates can do it, because radius is not a linear function of `x` and `y`.

Kernel PCA maps to a feature space where it is. Run both and check with a linear
classifier on the resulting components.

## The data (created in the next cell)

- `X` `(600, 2)`: two concentric noisy rings.
- `y` `(600,)`: inner ring 0, outer ring 1.

## Your task

With **sklearn and NumPy**:

1. Fit `PCA(n_components=2)` and `KernelPCA(n_components=3, kernel="rbf",
   gamma=0.5)` on `X`. Note the asymmetry: `X` has only two columns, so linear
   PCA cannot produce a third component, while kernel PCA can, and it needs to,
   because the radius information does not land entirely in the first one.
2. For each embedding, fit `LogisticRegression(max_iter=2000)` on the
   components and score training accuracy.
3. Also score a logistic model on the raw `X`.
4. Set `answers` to a dict with `"acc_raw"`, `"acc_pca"`, `"acc_kpca"`, and
   `"kpca_component_gap"`, the absolute difference in mean first-component value
   between the two classes, divided by that component's standard deviation
   (NumPy's `std` with its default `ddof=0`).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks kernel PCA is above 0.95 and linear PCA below 0.7.

In [ ]:
import numpy as np

rng = np.random.default_rng(3008)
n = 300
t1 = rng.uniform(0, 2 * np.pi, n)
t2 = rng.uniform(0, 2 * np.pi, n)
inner = np.column_stack([np.cos(t1), np.sin(t1)]) * 1.0 + rng.normal(0, 0.08, (n, 2))
outer = np.column_stack([np.cos(t2), np.sin(t2)]) * 2.2 + rng.normal(0, 0.08, (n, 2))
X = np.vstack([inner, outer])
y = np.concatenate([np.zeros(n, int), np.ones(n, int)])
print(f"X {X.shape}: rings at radius 1.0 and 2.2")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("acc_raw", [], 0.52, -0.08706776770353956, 1e-06),
    ("acc_pca", [], 0.52, -0.08706776770353956, 1e-06),
    ("acc_kpca", [], 1.0, -0.16743801481449916, 1e-06),
    ("kpca_component_gap", [], 0.1194784867452151, -0.020005240633659268, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if all(_k in answers for _k in ("acc_raw", "acc_pca", "acc_kpca")):
    _g = answers["acc_kpca"] > 0.95 and answers["acc_pca"] < 0.7
    print(f"  {'kernel PCA separates':<24} {'ok' if _g else 'wrong'}  "
          f"(raw {answers['acc_raw']:.3f}, pca {answers['acc_pca']:.3f}, "
          f"kpca {answers['acc_kpca']:.3f})")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Linear PCA can only rotate and rescale, and with two input columns it cannot
produce more than two components, so its output is the raw data in disguise, and
the identical accuracy is the proof. The two rings have the same centre, so
every linear projection mixes them. The RBF
kernel's feature map makes similarity fall off with distance, and since points on
the same ring are typically closer to each other than to the other ring, the
third kernel component ends up tracking radius, while the first two do not
separate the rings. This is the cleanest
demonstration that "dimensionality reduction" and "linear dimensionality
reduction" are different tools.

```python
import numpy as np
from sklearn.decomposition import PCA, KernelPCA
from sklearn.linear_model import LogisticRegression

def acc(Z):
    m = LogisticRegression(max_iter=2000).fit(Z, y)
    return float((m.predict(Z) == y).mean())

Zp = PCA(n_components=2).fit_transform(X)
Zk = KernelPCA(n_components=3, kernel="rbf", gamma=0.5).fit_transform(X)

c0 = Zk[y == 0, 0].mean()
c1 = Zk[y == 1, 0].mean()
gap = float(abs(c1 - c0) / Zk[:, 0].std())

answers = {
    "acc_raw": acc(X),
    "acc_pca": acc(Zp),
    "acc_kpca": acc(Zk),
    "kpca_component_gap": gap,
}
```